# Análise de características usando um Comitê de Classificadores

> **Autores:** Alexandre Maciel e Vinicius de Lima.

Esse notebook calcula o F1-score para diferentes cenários usando um comitê de classificadores para diferentes tamanhos para as bases geradas por [bases.ipynb](./bases.ipynb), com as características extraídas das raças de cachorro Samoieda e Pug e das raças de gato Russian Blue e Birman.

In [30]:
import warnings
import os
import pickle
import pandas as pd
import numpy as np
from itertools import cycle
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    StackingClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression
from scipy.stats import f_oneway
from scipy.stats import tukey_hsd

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
SCORING = "f1_macro"
BASES_DIR = "bases/"
RESULTS_DIR = "results/"
os.makedirs(RESULTS_DIR, exist_ok=True)

DATASETS = [
    "PCA_075_HOG_256_32x32.csv",
    "PCA_075_HOG_128_16x16.csv",
    "PCA_090_HOG_256_32x32.csv",
    "HOG_256_32x32.csv",
    "PCA_090_HOG_128_16x16.csv",
    "PCA_075_HOG_128_32x32.csv",
    "PCA_090_HOG_128_32x32.csv",
    "HOG_128_16x16.csv",
    "HOG_128_32x32.csv",
    "LBP_256_6R.csv",
    "PCA_075_HOG_256_16x16.csv",
    "LBP_256_12R.csv",
]

print("Configuração concluída.")


Configuração concluída.


In [31]:
def load_dataset(name):
    df = pd.read_csv(BASES_DIR + name)
    return df.iloc[:, :-1], df["label"]

all_data = {ds: load_dataset(ds) for ds in DATASETS}
print(f"{len(all_data)} bases carregadas.")


12 bases carregadas.


In [32]:
knn_results = pd.read_csv("knn/results.csv")
k_values = [f"{k}k" for k in range(1, 11)]
means = knn_results.loc[:, k_values].mean()
best_k = int(means.idxmax().replace("k", ""))
print(f"k médio por valor:\n{means.sort_values(ascending=False)}\
")
print(f"Melhor k para KNN: {best_k}")


k médio por valor:
10k    0.620691
9k     0.620136
7k     0.613391
8k     0.613222
5k     0.608632
3k     0.601061
6k     0.600769
4k     0.587301
1k     0.580522
2k     0.542231
dtype: float64
Melhor k para KNN: 10


In [33]:
def evaluate_cv(model, xs, ys):
    """Return (mean_f1, list_of_10_fold_scores)."""
    scores = cross_val_score(model, xs, ys, cv=10, scoring=SCORING, n_jobs=-1)
    return scores.mean(), scores

def evaluate_7030(model, xs, ys):
    """Return f1 on 70/30 split."""
    xs_train, xs_test, ys_train, ys_test = train_test_split(
        xs, ys, test_size=0.3, random_state=RANDOM_STATE
    )
    model.fit(xs_train, ys_train)
    return f1_score(ys_test, model.predict(xs_test), average="macro")

print("Funções de avaliação definidas.")


Funções de avaliação definidas.


In [34]:
best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_dt = DecisionTreeClassifier(
    criterion="gini", max_depth=10, random_state=RANDOM_STATE
)
best_nb = GaussianNB()
best_mlp = MLPClassifier(
    hidden_layer_sizes=59,
    activation="relu",
    solver="adam",
    learning_rate_init=0.0001,
    max_iter=2500,
    random_state=RANDOM_STATE,
    early_stopping=True,
)

BAGGING_ESTIMATORS = [
    ("k-NN", best_knn),
    ("DT", best_dt),
    ("NB", best_nb),
    ("MLP", best_mlp),
]
print("Configurações base definidas.")


Configurações base definidas.


## Bagging

Utilizando `BaggingClassifier` com 4 estimadores base (k-NN, DT, NB, MLP) e 3 valores de `n_estimators` (10, 15, 20).

In [35]:
BAGGING_CACHE = RESULTS_DIR + "bagging_results.pkl"
BAGGING_TABLE = RESULTS_DIR + "bagging_table.csv"

if os.path.exists(BAGGING_CACHE):
    with open(BAGGING_CACHE, "rb") as f:
        bagging_cv_scores, bagging_test_scores = pickle.load(f)
    print("Resultados do Bagging carregados do cache.")
else:
    bagging_cv_scores = {}  # (est_name, n) -> list of scores (all bases × folds)
    bagging_test_scores = {}  # (est_name, n) -> list of test scores per base

    n_est_list = [10, 15, 20]
    for est_name, est in BAGGING_ESTIMATORS:
        for n in n_est_list:
            key = (est_name, n)
            all_cv = []
            all_test = []
            for ds in DATASETS:
                xs, ys = all_data[ds]
                model = BaggingClassifier(
                    estimator=est, n_estimators=n,
                    random_state=RANDOM_STATE, n_jobs=-1,
                )
                cv_mean, cv_scores = evaluate_cv(model, xs, ys)
                all_cv.append(cv_scores)
                test_score = evaluate_7030(model, xs, ys)
                all_test.append(test_score)
            bagging_cv_scores[key] = np.concatenate(all_cv)
            bagging_test_scores[key] = all_test
            print(f"  Bagging {est_name}({n}) OK")

    with open(BAGGING_CACHE, "wb") as f:
        pickle.dump((bagging_cv_scores, bagging_test_scores), f)
    print("Cache salvo.")


  Bagging k-NN(10) OK
  Bagging k-NN(15) OK
  Bagging k-NN(20) OK
  Bagging DT(10) OK
  Bagging DT(15) OK
  Bagging DT(20) OK
  Bagging NB(10) OK
  Bagging NB(15) OK
  Bagging NB(20) OK
  Bagging MLP(10) OK
  Bagging MLP(15) OK
  Bagging MLP(20) OK
Cache salvo.


In [36]:
n_est_list = [10, 15, 20]
bagging_rows = []
for i, ds in enumerate(DATASETS, 1):
    base_name = ds.replace('.csv', '')
    xs, ys = all_data[ds]

    # 10-fold CV row
    row_cv = {'#': i, 'Base': base_name, 'Treinamento / Teste': '10-fold CV'}
    for est_name, _ in BAGGING_ESTIMATORS:
        for n in n_est_list:
            key = (est_name, n)
            # Extract scores for this specific base from the concatenated array
            base_idx = DATASETS.index(ds)
            start = base_idx * 10
            end = start + 10
            base_scores = bagging_cv_scores[key][start:end]
            row_cv[f'{est_name} ({n})'] = round(base_scores.mean(), 4)
    bagging_rows.append(row_cv)

    # 70/30 row
    row_7030 = {'#': i, 'Base': base_name, 'Treinamento / Teste': '70/30'}
    for est_name, _ in BAGGING_ESTIMATORS:
        for n in n_est_list:
            row_7030[f'{est_name} ({n})'] = round(bagging_test_scores[(est_name, n)][i-1], 4)
    bagging_rows.append(row_7030)

df_bagging = pd.DataFrame(bagging_rows)
df_bagging = df_bagging.set_index(['#', 'Base', 'Treinamento / Teste'])

# Add summary rows
data_cols_b = df_bagging.select_dtypes(include=[np.number]).columns
mean_row_b = {col: df_bagging[col].mean() for col in data_cols_b}
std_row_b = {col: df_bagging[col].std() for col in data_cols_b}

def make_summary_row(d, label):
    s = pd.Series(index=df_bagging.columns, dtype=float)
    for col in data_cols_b:
        s[col] = d[col]
    s.name = ('', label, '')
    return s

df_bagging = pd.concat([
    df_bagging,
    make_summary_row(mean_row_b, 'Média =>').to_frame().T,
    make_summary_row(std_row_b, 'Desv. Pad. =>').to_frame().T,
])

df_bagging.to_csv(BAGGING_TABLE)
df_bagging


k-NN (10)  k-NN (15)  k-NN (20)  \
1  PCA_075_HOG_256_32x32 10-fold CV   0.694800   0.710500   0.709300   
                         70/30        0.687100   0.686300   0.681900   
2  PCA_075_HOG_128_16x16 10-fold CV   0.684500   0.679000   0.683600   
                         70/30        0.683300   0.670800   0.683100   
3  PCA_090_HOG_256_32x32 10-fold CV   0.682200   0.691100   0.696100   
                         70/30        0.677100   0.672700   0.685500   
4  HOG_256_32x32         10-fold CV   0.687500   0.692000   0.697000   
                         70/30        0.655900   0.676700   0.681900   
5  PCA_090_HOG_128_16x16 10-fold CV   0.645100   0.644600   0.658900   
                         70/30        0.645800   0.637000   0.624300   
6  PCA_075_HOG_128_32x32 10-fold CV   0.670000   0.671200   0.664800   
                         70/30        0.662100   0.656000   0.659800   
7  PCA_090_HOG_128_32x32 10-fold CV   0.648200   0.658300   0.662100   
                         70/30        0.630500   0.646700   0.641900   
8  HOG_128_16x16         10-fold CV   0.644000   0.633900   0.639600   
                         70/30        0.637400   0.629100   0.633100   
9  HOG_128_32x32         10-fold CV   0.632700   0.632800   0.640000   
                         70/30        0.615400   0.653300   0.650500   
10 LBP_256_6R            10-fold CV   0.553300   0.558800   0.563600   
                         70/30        0.612400   0.644800   0.657200   
11 PCA_075_HOG_256_16x16 10-fold CV   0.630600   0.649900   0.657300   
                         70/30        0.623000   0.623000   0.622900   
12 LBP_256_12R           10-fold CV   0.539800   0.556200   0.546300   
                         70/30        0.583300   0.595300   0.595700   
   Média =>                           0.642750   0.648750   0.651517   
   Desv. Pad. =>                      0.041051   0.038179   0.040154   

                                      DT (10)   DT (15)   DT (20)   NB (10)  \
1  PCA_075_HOG_256_32x32 10-fold CV  0.612400  0.631400  0.649200  0.695000   
                         70/30       0.652800  0.666600  0.666500  0.683000   
2  PCA_075_HOG_128_16x16 10-fold CV  0.634200  0.652800  0.663600  0.677100   
                         70/30       0.657200  0.679200  0.649900  0.655400   
3  PCA_090_HOG_256_32x32 10-fold CV  0.587800  0.605000  0.605500  0.638600   
                         70/30       0.599900  0.641600  0.645800  0.654100   
4  HOG_256_32x32         10-fold CV  0.662800  0.685900  0.673800  0.703300   
                         70/30       0.699700  0.691600  0.700000  0.737400   
5  PCA_090_HOG_128_16x16 10-fold CV  0.630000  0.625900  0.653400  0.639600   
                         70/30       0.624900  0.645300  0.641400  0.620000   
6  PCA_075_HOG_128_32x32 10-fold CV  0.618000  0.633900  0.653300  0.697500   
                         70/30       0.683300  0.694900  0.691600  0.737500   
7  PCA_090_HOG_128_32x32 10-fold CV  0.650200  0.631200  0.657800  0.680200   
                         70/30       0.662400  0.675000  0.654000  0.691100   
8  HOG_128_16x16         10-fold CV  0.647200  0.688400  0.684100  0.732200   
                         70/30       0.631700  0.645800  0.629000  0.749800   
9  HOG_128_32x32         10-fold CV  0.638600  0.684700  0.674400  0.704200   
                         70/30       0.628600  0.641600  0.654200  0.737300   
10 LBP_256_6R            10-fold CV  0.524900  0.537600  0.549900  0.594000   
                         70/30       0.574500  0.616200  0.607700  0.602600   
11 PCA_075_HOG_256_16x16 10-fold CV  0.653800  0.673100  0.691700  0.622600   
                         70/30       0.616600  0.658300  0.662500  0.632100   
12 LBP_256_12R           10-fold CV  0.544500  0.567100  0.559000  0.594100   
                         70/30       0.587500  0.620700  0.587300  0.600500   
   Média =>                          0.625979  0.645575  0.646067  0.669967   
   Desv. Pad. =>                     0

### Análise — Bagging

Comparando os 12 modelos (4 estimadores × 3 `n_estimators`) via ANOVA + Tukey HSD.

In [37]:
n_est_list = [10, 15, 20]
bagging_model_names = []
bagging_model_groups = []
for est_name, _ in BAGGING_ESTIMATORS:
    for n in n_est_list:
        name = f'{est_name}({n})'
        bagging_model_names.append(name)
        bagging_model_groups.append(bagging_cv_scores[(est_name, n)])

if any(np.std(g) < 1e-10 for g in bagging_model_groups):
    print("AVISO: Variância zero detectada.")
else:
    f_stat, p_val = f_oneway(*bagging_model_groups)
    print(f"ANOVA: F={f_stat:.4f}, p={p_val:.6f}")

    if p_val < 0.05:
        print("Diferença significativa encontrada. Aplicando Tukey HSD...")
        res = tukey_hsd(*bagging_model_groups)
        means = {f'{e}({n_})': np.mean(bagging_cv_scores[(e, n_)])
                 for e, _ in BAGGING_ESTIMATORS for n_ in n_est_list}
        best_name = max(means, key=means.get)
        best_idx = bagging_model_names.index(best_name)
        # Check if significantly better than second best
        sorted_means = sorted(means, key=means.get, reverse=True)
        second_name = sorted_means[1]
        second_idx = bagging_model_names.index(second_name)
        if abs(res.pvalue[best_idx, second_idx]) < 0.05:
            print(f"Melhor modelo: {best_name} (F1={means[best_name]:.4f})")
        else:
            print("Melhor modelo não é significativamente superior ao segundo. "
                  "Escolhendo por complexidade...")
            # Complexity: fewer n_estimators and simpler estimator type
            complexity_order = {'NB': 0, 'k-NN': 1, 'DT': 2, 'MLP': 3}
            def comp(name):
                e, n = name.split('(')
                n = int(n.rstrip(')'))
                return complexity_order.get(e, 99) * 100 + n
            # Candidates not significantly different from best
            candidates = [
                bagging_model_names[i] for i in range(len(bagging_model_names))
                if abs(res.pvalue[best_idx, i]) >= 0.05
            ]
            best_name = min(candidates, key=comp)
            print(f"Escolhido: {best_name}")
    else:
        print("Sem diferença significativa. Escolhendo por complexidade...")
        complexity_order = {'NB': 0, 'k-NN': 1, 'DT': 2, 'MLP': 3}
        def comp(name):
            e, n = name.split('(')
            n = int(n.rstrip(')'))
            return complexity_order.get(e, 99) * 100 + n
        means = {f'{e}({n_})': np.mean(bagging_cv_scores[(e, n_)])
                 for e, _ in BAGGING_ESTIMATORS for n_ in n_est_list}
        best_mean_val = max(means.values())
        candidates = [n for n, v in means.items() if abs(v - best_mean_val) < 0.01]
        best_name = min(candidates, key=comp)
        print(f"Escolhido: {best_name} (F1={means[best_name]:.4f})")


ANOVA: F=62.5774, p=0.000000
Diferença significativa encontrada. Aplicando Tukey HSD...
Melhor modelo não é significativamente superior ao segundo. Escolhendo por complexidade...
Escolhido: NB(10)


## Random Forest

Utilizando `RandomForestClassifier` com 2 critérios (gini, entropy) e 4 valores de `n_estimators` (10, 20, 30, 100).

In [38]:
RF_CACHE = RESULTS_DIR + "rf_results.pkl"
RF_TABLE = RESULTS_DIR + "rf_table.csv"

RF_CRITERIA = ['gini', 'entropy']
RF_N_ESTIMATORS = [10, 20, 30, 100]

if os.path.exists(RF_CACHE):
    with open(RF_CACHE, "rb") as f:
        rf_cv_scores, rf_test_scores = pickle.load(f)
    print("Resultados do RF carregados do cache.")
else:
    rf_cv_scores = {}
    rf_test_scores = {}
    for criterion in RF_CRITERIA:
        for n in RF_N_ESTIMATORS:
            key = (criterion, n)
            all_cv = []
            all_test = []
            for ds in DATASETS:
                xs, ys = all_data[ds]
                model = RandomForestClassifier(
                    criterion=criterion, n_estimators=n,
                    random_state=RANDOM_STATE, n_jobs=-1,
                )
                cv_mean, cv_scores = evaluate_cv(model, xs, ys)
                all_cv.append(cv_scores)
                test_score = evaluate_7030(model, xs, ys)
                all_test.append(test_score)
            rf_cv_scores[key] = np.concatenate(all_cv)
            rf_test_scores[key] = all_test
            print(f"  RF {criterion}({n}) OK")

    with open(RF_CACHE, "wb") as f:
        pickle.dump((rf_cv_scores, rf_test_scores), f)
    print("Cache salvo.")


  RF gini(10) OK
  RF gini(20) OK
  RF gini(30) OK
  RF gini(100) OK
  RF entropy(10) OK
  RF entropy(20) OK
  RF entropy(30) OK
  RF entropy(100) OK
Cache salvo.


In [39]:
rf_rows = []
for i, ds in enumerate(DATASETS, 1):
    base_name = ds.replace('.csv', '')

    row_cv = {'#': i, 'Base': base_name, 'Treinamento / Teste': '10-fold CV'}
    for crit in RF_CRITERIA:
        for n in RF_N_ESTIMATORS:
            key = (crit, n)
            base_idx = DATASETS.index(ds)
            base_scores = rf_cv_scores[key][base_idx * 10:(base_idx + 1) * 10]
            row_cv[f'{crit.capitalize()} ({n})'] = round(base_scores.mean(), 4)
    rf_rows.append(row_cv)

    row_7030 = {'#': i, 'Base': base_name, 'Treinamento / Teste': '70/30'}
    for crit in RF_CRITERIA:
        for n in RF_N_ESTIMATORS:
            row_7030[f'{crit.capitalize()} ({n})'] = round(rf_test_scores[(crit, n)][i-1], 4)
    rf_rows.append(row_7030)

df_rf = pd.DataFrame(rf_rows)
df_rf = df_rf.set_index(['#', 'Base', 'Treinamento / Teste'])

data_cols_rf = df_rf.select_dtypes(include=[np.number]).columns
mean_row_rf = {col: df_rf[col].mean() for col in data_cols_rf}
std_row_rf = {col: df_rf[col].std() for col in data_cols_rf}

def make_summary_row(d, label):
    s = pd.Series(index=df_rf.columns, dtype=float)
    for col in data_cols_rf:
        s[col] = d[col]
    s.name = ('', label, '')
    return s

df_rf = pd.concat([
    df_rf,
    make_summary_row(mean_row_rf, 'Média =>').to_frame().T,
    make_summary_row(std_row_rf, 'Desv. Pad. =>').to_frame().T,
])

df_rf.to_csv(RF_TABLE)
df_rf


Gini (10)  Gini (20)  Gini (30)  \
1  PCA_075_HOG_256_32x32 10-fold CV   0.632000   0.647200   0.644200   
                         70/30        0.612800   0.657500   0.679000   
2  PCA_075_HOG_128_16x16 10-fold CV   0.595500   0.631100   0.682700   
                         70/30        0.598200   0.637200   0.654000   
3  PCA_090_HOG_256_32x32 10-fold CV   0.564600   0.585700   0.582300   
                         70/30        0.591800   0.622400   0.597700   
4  HOG_256_32x32         10-fold CV   0.661200   0.688800   0.691800   
                         70/30        0.628400   0.686600   0.653700   
5  PCA_090_HOG_128_16x16 10-fold CV   0.549600   0.577000   0.589900   
                         70/30        0.539100   0.570600   0.599600   
6  PCA_075_HOG_128_32x32 10-fold CV   0.625700   0.656700   0.669800   
                         70/30        0.625700   0.682300   0.711700   
7  PCA_090_HOG_128_32x32 10-fold CV   0.575800   0.636700   0.650900   
                         70/30        0.600500   0.641300   0.683300   
8  HOG_128_16x16         10-fold CV   0.604600   0.650400   0.658100   
                         70/30        0.626300   0.670600   0.707800   
9  HOG_128_32x32         10-fold CV   0.631500   0.654700   0.668300   
                         70/30        0.679600   0.711900   0.745300   
10 LBP_256_6R            10-fold CV   0.541100   0.547500   0.548300   
                         70/30        0.599000   0.587300   0.591200   
11 PCA_075_HOG_256_16x16 10-fold CV   0.580100   0.625300   0.631400   
                         70/30        0.652900   0.630800   0.652400   
12 LBP_256_12R           10-fold CV   0.574500   0.563000   0.567300   
                         70/30        0.584900   0.599300   0.616700   
   Média =>                           0.603142   0.631746   0.644892   
   Desv. Pad. =>                      0.036467   0.043307   0.049694   

                                     Gini (100)  Entropy (10)  Entropy (20)  \
1  PCA_075_HOG_256_32x32 10-fold CV    0.693400      0.618700      0.636100   
                         70/30         0.695800      0.649700      0.631300   
2  PCA_075_HOG_128_16x16 10-fold CV    0.697100      0.590100      0.625800   
                         70/30         0.694600      0.575900      0.613400   
3  PCA_090_HOG_256_32x32 10-fold CV    0.658200      0.580400      0.612600   
                         70/30         0.683000      0.627300      0.633100   
4  HOG_256_32x32         10-fold CV    0.723200      0.652900      0.687700   
                         70/30         0.712500      0.689100      0.728800   
5  PCA_090_HOG_128_16x16 10-fold CV    0.639000      0.573800      0.587800   
                         70/30         0.662200      0.461900      0.562300   
6  PCA_075_HOG_128_32x32 10-fold CV    0.703500      0.631400      0.638500   
                         70/30         0.737300      0.656400      0.655400   
7  PCA_090_HOG_128_32x32 10-fold CV    0.697100      0.592700      0.623900   
                         70/30         0.754200      0.592800      0.663300   
8  HOG_128_16x16         10-fold CV    0.720700      0.640200      0.684300   
                         70/30         0.708300      0.635700      0.641000   
9  HOG_128_32x32         10-fold CV    0.703000      0.644300      0.683400   
                         70/30         0.724800      0.685900      0.682300   
10 LBP_256_6R            10-fold CV    0.553900      0.524400      0.540100   
                         70/30         0.616600      0.582300      0.620800   
11 PCA_075_HOG_256_16x16 10-fold CV    0.689700      0.564400      0.607600   
                         70/30         0.662500      0.615200      0.695600   
12 LBP_256_12R           10-fold CV    0.567900      0.556300      0.575900   
                         70/30         0.604000      0.602200      0.616200   
   Média =>                            0.679271      0.606000      0.635300   
   Desv. Pad. =>                      

### Análise — Random Forest

In [40]:
rf_model_names = []
rf_model_groups = []
for crit in RF_CRITERIA:
    for n in RF_N_ESTIMATORS:
        name = f'{crit}({n})'
        rf_model_names.append(name)
        rf_model_groups.append(rf_cv_scores[(crit, n)])

if any(np.std(g) < 1e-10 for g in rf_model_groups):
    print("AVISO: Variância zero detectada.")
else:
    f_stat, p_val = f_oneway(*rf_model_groups)
    print(f"ANOVA: F={f_stat:.4f}, p={p_val:.6f}")

    if p_val < 0.05:
        print("Diferença significativa. Aplicando Tukey HSD...")
        res = tukey_hsd(*rf_model_groups)
        means = {f'{c}({n_})': np.mean(rf_cv_scores[(c, n_)])
                 for c in RF_CRITERIA for n_ in RF_N_ESTIMATORS}
        best_name = max(means, key=means.get)
        best_idx = rf_model_names.index(best_name)
        sorted_means = sorted(means, key=means.get, reverse=True)
        second_name = sorted_means[1]
        second_idx = rf_model_names.index(second_name)
        if abs(res.pvalue[best_idx, second_idx]) < 0.05:
            print(f"Melhor modelo: {best_name} (F1={means[best_name]:.4f})")
        else:
            print("Melhor não difere significativamente do segundo. "
                  "Escolhendo por complexidade...")
            def comp(name):
                c, n = name.split('(')
                n = int(n.rstrip(')'))
                return (0 if c == 'gini' else 1) * 100 + n
            candidates = [
                rf_model_names[i] for i in range(len(rf_model_names))
                if abs(res.pvalue[best_idx, i]) >= 0.05
            ]
            best_name = min(candidates, key=comp)
            print(f"Escolhido: {best_name}")
    else:
        print("Sem diferença significativa. Escolhendo por complexidade...")
        def comp(name):
            c, n = name.split('(')
            n = int(n.rstrip(')'))
            return (0 if c == 'gini' else 1) * 100 + n
        means = {f'{c}({n_})': np.mean(rf_cv_scores[(c, n_)])
                 for c in RF_CRITERIA for n_ in RF_N_ESTIMATORS}
        best_mean_val = max(means.values())
        candidates = [n for n, v in means.items() if abs(v - best_mean_val) < 0.01]
        best_name = min(candidates, key=comp)
        print(f"Escolhido: {best_name} (F1={means[best_name]:.4f})")


ANOVA: F=16.4842, p=0.000000
Diferença significativa. Aplicando Tukey HSD...
Melhor não difere significativamente do segundo. Escolhendo por complexidade...
Escolhido: gini(100)


## Funções auxiliares para gerar comitês heterogêneos

Para Stacking e Voting, geramos `N` classificadores base distribuindo ciclicamente entre k-NN, DT, NB e MLP, com hiperparâmetros variados para diversidade.

In [41]:
def make_knn(i):
    ks = [3, 5, 7, 9]
    return KNeighborsClassifier(n_neighbors=ks[i % len(ks)])

def make_dt(i):
    depths = [4, 6, 8, 10]
    return DecisionTreeClassifier(
        criterion='gini', max_depth=depths[i % len(depths)],
        random_state=RANDOM_STATE + i
    )

def make_nb(i):
    return GaussianNB()

def make_simple_mlp(i):
    """MLP leve para Voting (conforme especificação, evitar configs complexas)."""
    return MLPClassifier(
        hidden_layer_sizes=(10,), max_iter=500,
        early_stopping=True,
        random_state=RANDOM_STATE + i
    )

def make_best_mlp(i):
    """MLP com a melhor configuração encontrada para Stacking."""
    return MLPClassifier(
        hidden_layer_sizes=59, activation='relu', solver='adam',
        learning_rate_init=0.0001, max_iter=2500,
        early_stopping=True,
        random_state=RANDOM_STATE + i
    )

def generate_committee(n, mlp_maker):
    makers = cycle([make_knn, make_dt, make_nb, mlp_maker])
    estimators = []
    for i in range(n):
        maker = next(makers)
        estimators.append((f'clf_{i}', maker(i)))
    return estimators


## Stacking

Utilizando `StackingClassifier` com `LogisticRegression` como meta-classificador e comitês de 5, 10, 15 e 20 classificadores base.

In [42]:
STACKING_CACHE = RESULTS_DIR + "stacking_results.pkl"
STACKING_TABLE = RESULTS_DIR + "stacking_table.csv"

STACKING_N_VALUES = [5, 10, 15, 20]

if os.path.exists(STACKING_CACHE):
    with open(STACKING_CACHE, "rb") as f:
        stacking_cv_scores, stacking_test_scores = pickle.load(f)
    print("Resultados do Stacking carregados do cache.")
else:
    stacking_cv_scores = {}
    stacking_test_scores = {}
    for n in STACKING_N_VALUES:
        key = n
        all_cv = []
        all_test = []
        for ds in DATASETS:
            xs, ys = all_data[ds]
            estimators = generate_committee(n, make_best_mlp)
            model = StackingClassifier(
                estimators=estimators,
                final_estimator=LogisticRegression(random_state=RANDOM_STATE),
                cv=3,
                n_jobs=-1,
            )
            cv_mean, cv_scores = evaluate_cv(model, xs, ys)
            all_cv.append(cv_scores)
            test_score = evaluate_7030(model, xs, ys)
            all_test.append(test_score)
        stacking_cv_scores[key] = np.concatenate(all_cv)
        stacking_test_scores[key] = all_test
        print(f"  Stacking (n={n}) OK")

    with open(STACKING_CACHE, "wb") as f:
        pickle.dump((stacking_cv_scores, stacking_test_scores), f)
    print("Cache salvo.")


  Stacking (n=5) OK
  Stacking (n=10) OK
  Stacking (n=15) OK
  Stacking (n=20) OK
Cache salvo.


In [43]:
stacking_rows = []
for i, ds in enumerate(DATASETS, 1):
    base_name = ds.replace('.csv', '')

    row_cv = {'#': i, 'Base': base_name, 'Treinamento / Teste': '10-fold CV'}
    for n in STACKING_N_VALUES:
        base_idx = DATASETS.index(ds)
        base_scores = stacking_cv_scores[n][base_idx * 10:(base_idx + 1) * 10]
        row_cv[f'Classificadores ({n})'] = round(base_scores.mean(), 4)
    stacking_rows.append(row_cv)

    row_7030 = {'#': i, 'Base': base_name, 'Treinamento / Teste': '70/30'}
    for n in STACKING_N_VALUES:
        row_7030[f'Classificadores ({n})'] = round(stacking_test_scores[n][i-1], 4)
    stacking_rows.append(row_7030)

df_stacking = pd.DataFrame(stacking_rows)
df_stacking = df_stacking.set_index(['#', 'Base', 'Treinamento / Teste'])

data_cols_s = df_stacking.select_dtypes(include=[np.number]).columns
mean_row_s = {col: df_stacking[col].mean() for col in data_cols_s}
std_row_s = {col: df_stacking[col].std() for col in data_cols_s}

def make_summary_row(d, label):
    s = pd.Series(index=df_stacking.columns, dtype=float)
    for col in data_cols_s:
        s[col] = d[col]
    s.name = ('', label, '')
    return s

df_stacking = pd.concat([
    df_stacking,
    make_summary_row(mean_row_s, 'Média =>').to_frame().T,
    make_summary_row(std_row_s, 'Desv. Pad. =>').to_frame().T,
])

df_stacking.to_csv(STACKING_TABLE)
df_stacking


Classificadores (5)  \
1  PCA_075_HOG_256_32x32 10-fold CV             0.714300   
                         70/30                  0.741600   
2  PCA_075_HOG_128_16x16 10-fold CV             0.724500   
                         70/30                  0.677900   
3  PCA_090_HOG_256_32x32 10-fold CV             0.684000   
                         70/30                  0.754100   
4  HOG_256_32x32         10-fold CV             0.711300   
                         70/30                  0.754100   
5  PCA_090_HOG_128_16x16 10-fold CV             0.705700   
                         70/30                  0.670400   
6  PCA_075_HOG_128_32x32 10-fold CV             0.696000   
                         70/30                  0.695700   
7  PCA_090_HOG_128_32x32 10-fold CV             0.704700   
                         70/30                  0.700000   
8  HOG_128_16x16         10-fold CV             0.720300   
                         70/30                  0.724700   
9  HOG_128_32x32         10-fold CV             0.693000   
                         70/30                  0.737300   
10 LBP_256_6R            10-fold CV             0.565300   
                         70/30                  0.611400   
11 PCA_075_HOG_256_16x16 10-fold CV             0.662900   
                         70/30                  0.648400   
12 LBP_256_12R           10-fold CV             0.584300   
                         70/30                  0.581000   
   Média =>                                     0.685954   
   Desv. Pad. =>                                0.053379   

                                     Classificadores (10)  \
1  PCA_075_HOG_256_32x32 10-fold CV              0.708000   
                         70/30                   0.737500   
2  PCA_075_HOG_128_16x16 10-fold CV              0.713600   
                         70/30                   0.681500   
3  PCA_090_HOG_256_32x32 10-fold CV              0.700500   
                         70/30                   0.754200   
4  HOG_256_32x32         10-fold CV              0.720400   
                         70/30                   0.758200   
5  PCA_090_HOG_128_16x16 10-fold CV              0.685600   
                         70/30                   0.637200   
6  PCA_075_HOG_128_32x32 10-fold CV              0.690200   
                         70/30                   0.704100   
7  PCA_090_HOG_128_32x32 10-fold CV              0.693800   
                         70/30                   0.695800   
8  HOG_128_16x16         10-fold CV              0.714000   
                         70/30                   0.724700   
9  HOG_128_32x32         10-fold CV              0.694300   
                         70/30                   0.737300   
10 LBP_256_6R            10-fold CV              0.565400   
                         70/30                   0.598600   
11 PCA_075_HOG_256_16x16 10-fold CV              0.667800   
                         70/30                   0.649100   
12 LBP_256_12R           10-fold CV              0.579800   
                         70/30                   0.577900   
   Média =>                                      0.682896   
   Desv. Pad. =>                                 0.055030   

                                     Classificadores (15)  \
1  PCA_075_HOG_256_32x32 10-fold CV              0.709100   
                         70/30                   0.720800   
2  PCA_075_HOG_128_16x16 10-fold CV              0.714000   
                         70/30                   0.672700   
3  PCA_090_HOG_256_32x32 10-fold CV              0.684200   
                         70/30                   0.754200   
4  HOG_256_32x32         10-fold CV              0.716700   
                         70/30                   0.758200   
5  PCA_090_HOG_128_16x16 10-fold CV              0.690500   
                         70/30                   0.641300   
6  PCA_075_HOG_128_32x32 10-fold CV              0.689000   
                         70/30      

### Análise — Stacking

In [44]:
stacking_model_names = [f'Stacking({n})' for n in STACKING_N_VALUES]
stacking_model_groups = [stacking_cv_scores[n] for n in STACKING_N_VALUES]

if any(np.std(g) < 1e-10 for g in stacking_model_groups):
    print("AVISO: Variância zero detectada.")
else:
    f_stat, p_val = f_oneway(*stacking_model_groups)
    print(f"ANOVA: F={f_stat:.4f}, p={p_val:.6f}")

    if p_val < 0.05:
        print("Diferença significativa. Aplicando Tukey HSD...")
        res = tukey_hsd(*stacking_model_groups)
        means = {n: np.mean(stacking_cv_scores[n]) for n in STACKING_N_VALUES}
        best_n = max(means, key=means.get)
        best_idx = STACKING_N_VALUES.index(best_n)
        sorted_ns = sorted(STACKING_N_VALUES, key=lambda n: means[n], reverse=True)
        second_n = sorted_ns[1]
        second_idx = STACKING_N_VALUES.index(second_n)
        if abs(res.pvalue[best_idx, second_idx]) < 0.05:
            print(f"Melhor: Stacking({best_n}) (F1={means[best_n]:.4f})")
        else:
            print("Escolhendo por complexidade (menor n)...")
            candidates = [n for i, n in enumerate(STACKING_N_VALUES)
                          if abs(res.pvalue[best_idx, i]) >= 0.05]
            best_n = min(candidates)
            print(f"Escolhido: Stacking({best_n}) (F1={means[best_n]:.4f})")
    else:
        print("Sem diferença significativa. Escolhendo o mais simples (menor n).")
        means = {n: np.mean(stacking_cv_scores[n]) for n in STACKING_N_VALUES}
        best_mean_val = max(means.values())
        candidates = [n for n, v in means.items() if abs(v - best_mean_val) < 0.01]
        best_n = min(candidates)
        print(f"Escolhido: Stacking({best_n}) (F1={means[best_n]:.4f})")


ANOVA: F=0.0299, p=0.993046
Sem diferença significativa. Escolhendo o mais simples (menor n).
Escolhido: Stacking(5) (F1=0.6805)


## Voting

Utilizando `VotingClassifier` (votação _soft_) com comitês de 5, 10, 15 e 20 classificadores base. Conforme a especificação, MLP usa configuração leve para evitar tempo excessivo.

In [45]:
VOTING_CACHE = RESULTS_DIR + "voting_results.pkl"
VOTING_TABLE = RESULTS_DIR + "voting_table.csv"

VOTING_N_VALUES = [5, 10, 15, 20]

if os.path.exists(VOTING_CACHE):
    with open(VOTING_CACHE, "rb") as f:
        voting_cv_scores, voting_test_scores = pickle.load(f)
    print("Resultados do Voting carregados do cache.")
else:
    voting_cv_scores = {}
    voting_test_scores = {}
    for n in VOTING_N_VALUES:
        key = n
        all_cv = []
        all_test = []
        for ds in DATASETS:
            xs, ys = all_data[ds]
            # For Voting, use simple MLP (light config)
            estimators = generate_committee(n, make_simple_mlp)
            model = VotingClassifier(
                estimators=estimators,
                voting='soft',
                n_jobs=-1,
            )
            cv_mean, cv_scores = evaluate_cv(model, xs, ys)
            all_cv.append(cv_scores)
            test_score = evaluate_7030(model, xs, ys)
            all_test.append(test_score)
        voting_cv_scores[key] = np.concatenate(all_cv)
        voting_test_scores[key] = all_test
        print(f"  Voting (n={n}) OK")

    with open(VOTING_CACHE, "wb") as f:
        pickle.dump((voting_cv_scores, voting_test_scores), f)
    print("Cache salvo.")


  Voting (n=5) OK
  Voting (n=10) OK
  Voting (n=15) OK
  Voting (n=20) OK
Cache salvo.


In [46]:
voting_rows = []
for i, ds in enumerate(DATASETS, 1):
    base_name = ds.replace('.csv', '')

    row_cv = {'#': i, 'Base': base_name, 'Treinamento / Teste': '10-fold CV'}
    for n in VOTING_N_VALUES:
        base_idx = DATASETS.index(ds)
        base_scores = voting_cv_scores[n][base_idx * 10:(base_idx + 1) * 10]
        row_cv[f'Classificadores ({n})'] = round(base_scores.mean(), 4)
    voting_rows.append(row_cv)

    row_7030 = {'#': i, 'Base': base_name, 'Treinamento / Teste': '70/30'}
    for n in VOTING_N_VALUES:
        row_7030[f'Classificadores ({n})'] = round(voting_test_scores[n][i-1], 4)
    voting_rows.append(row_7030)

df_voting = pd.DataFrame(voting_rows)
df_voting = df_voting.set_index(['#', 'Base', 'Treinamento / Teste'])

data_cols_v = df_voting.select_dtypes(include=[np.number]).columns
mean_row_v = {col: df_voting[col].mean() for col in data_cols_v}
std_row_v = {col: df_voting[col].std() for col in data_cols_v}

def make_summary_row(d, label):
    s = pd.Series(index=df_voting.columns, dtype=float)
    for col in data_cols_v:
        s[col] = d[col]
    s.name = ('', label, '')
    return s

df_voting = pd.concat([
    df_voting,
    make_summary_row(mean_row_v, 'Média =>').to_frame().T,
    make_summary_row(std_row_v, 'Desv. Pad. =>').to_frame().T,
])

df_voting.to_csv(VOTING_TABLE)
df_voting


Classificadores (5)  \
1  PCA_075_HOG_256_32x32 10-fold CV             0.705900   
                         70/30                  0.712800   
2  PCA_075_HOG_128_16x16 10-fold CV             0.719500   
                         70/30                  0.658100   
3  PCA_090_HOG_256_32x32 10-fold CV             0.697200   
                         70/30                  0.724100   
4  HOG_256_32x32         10-fold CV             0.714200   
                         70/30                  0.729000   
5  PCA_090_HOG_128_16x16 10-fold CV             0.692900   
                         70/30                  0.610100   
6  PCA_075_HOG_128_32x32 10-fold CV             0.660200   
                         70/30                  0.697900   
7  PCA_090_HOG_128_32x32 10-fold CV             0.674400   
                         70/30                  0.707800   
8  HOG_128_16x16         10-fold CV             0.700800   
                         70/30                  0.691600   
9  HOG_128_32x32         10-fold CV             0.676100   
                         70/30                  0.698700   
10 LBP_256_6R            10-fold CV             0.578000   
                         70/30                  0.585900   
11 PCA_075_HOG_256_16x16 10-fold CV             0.623700   
                         70/30                  0.612500   
12 LBP_256_12R           10-fold CV             0.559600   
                         70/30                  0.589800   
   Média =>                                     0.667533   
   Desv. Pad. =>                                0.052331   

                                     Classificadores (10)  \
1  PCA_075_HOG_256_32x32 10-fold CV              0.706000   
                         70/30                   0.699800   
2  PCA_075_HOG_128_16x16 10-fold CV              0.708200   
                         70/30                   0.666300   
3  PCA_090_HOG_256_32x32 10-fold CV              0.690800   
                         70/30                   0.695600   
4  HOG_256_32x32         10-fold CV              0.720400   
                         70/30                   0.720700   
5  PCA_090_HOG_128_16x16 10-fold CV              0.685000   
                         70/30                   0.651500   
6  PCA_075_HOG_128_32x32 10-fold CV              0.660500   
                         70/30                   0.695200   
7  PCA_090_HOG_128_32x32 10-fold CV              0.663200   
                         70/30                   0.666700   
8  HOG_128_16x16         10-fold CV              0.708800   
                         70/30                   0.699900   
9  HOG_128_32x32         10-fold CV              0.692900   
                         70/30                   0.699200   
10 LBP_256_6R            10-fold CV              0.588800   
                         70/30                   0.591200   
11 PCA_075_HOG_256_16x16 10-fold CV              0.663900   
                         70/30                   0.612600   
12 LBP_256_12R           10-fold CV              0.566500   
                         70/30                   0.565200   
   Média =>                                      0.667454   
   Desv. Pad. =>                                 0.047848   

                                     Classificadores (15)  \
1  PCA_075_HOG_256_32x32 10-fold CV              0.704200   
                         70/30                   0.703500   
2  PCA_075_HOG_128_16x16 10-fold CV              0.722000   
                         70/30                   0.665500   
3  PCA_090_HOG_256_32x32 10-fold CV              0.701300   
                         70/30                   0.737400   
4  HOG_256_32x32         10-fold CV              0.724400   
                         70/30                   0.729200   
5  PCA_090_HOG_128_16x16 10-fold CV              0.705900   
                         70/30                   0.656800   
6  PCA_075_HOG_128_32x32 10-fold CV              0.666900   
                         70/30      

### Análise — Voting

In [47]:
voting_model_names = [f'Voting({n})' for n in VOTING_N_VALUES]
voting_model_groups = [voting_cv_scores[n] for n in VOTING_N_VALUES]

if any(np.std(g) < 1e-10 for g in voting_model_groups):
    print("AVISO: Variância zero detectada.")
else:
    f_stat, p_val = f_oneway(*voting_model_groups)
    print(f"ANOVA: F={f_stat:.4f}, p={p_val:.6f}")

    if p_val < 0.05:
        print("Diferença significativa. Aplicando Tukey HSD...")
        res = tukey_hsd(*voting_model_groups)
        means = {n: np.mean(voting_cv_scores[n]) for n in VOTING_N_VALUES}
        best_n = max(means, key=means.get)
        best_idx = VOTING_N_VALUES.index(best_n)
        sorted_ns = sorted(VOTING_N_VALUES, key=lambda n: means[n], reverse=True)
        second_n = sorted_ns[1]
        second_idx = VOTING_N_VALUES.index(second_n)
        if abs(res.pvalue[best_idx, second_idx]) < 0.05:
            print(f"Melhor: Voting({best_n}) (F1={means[best_n]:.4f})")
        else:
            print("Escolhendo por complexidade (menor n)...")
            candidates = [n for i, n in enumerate(VOTING_N_VALUES)
                          if abs(res.pvalue[best_idx, i]) >= 0.05]
            best_n = min(candidates)
            print(f"Escolhido: Voting({best_n}) (F1={means[best_n]:.4f})")
    else:
        print("Sem diferença significativa. Escolhendo o mais simples (menor n).")
        means = {n: np.mean(voting_cv_scores[n]) for n in VOTING_N_VALUES}
        best_mean_val = max(means.values())
        candidates = [n for n, v in means.items() if abs(v - best_mean_val) < 0.01]
        best_n = min(candidates)
        print(f"Escolhido: Voting({best_n}) (F1={means[best_n]:.4f})")


ANOVA: F=0.6918, p=0.557369
Sem diferença significativa. Escolhendo o mais simples (menor n).
Escolhido: Voting(15) (F1=0.6795)


## Análise Geral

Comparando os melhores modelos de cada estrutura de comitê (Bagging, Random Forest, Stacking e Voting) para determinar o melhor modelo global.

In [48]:
# Collect the best model from each method
# Bagging
n_est_list = [10, 15, 20]
bagging_means = {
    f'{est_name}({n})': np.mean(bagging_cv_scores[(est_name, n)])
    for est_name, _ in BAGGING_ESTIMATORS for n in n_est_list
}
best_bagging_key = max(bagging_means, key=bagging_means.get)
# Parse key
best_bagging_est = best_bagging_key.split('(')[0]
best_bagging_n = int(best_bagging_key.rstrip(')').split('(')[1])
# Find matching estimator
best_bagging_est_obj = None
for name, obj in BAGGING_ESTIMATORS:
    if name == best_bagging_est:
        best_bagging_est_obj = obj
        break
best_bagging_model = BaggingClassifier(
    estimator=best_bagging_est_obj,
    n_estimators=best_bagging_n,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# RF
rf_means = {
    f'{c}({n})': np.mean(rf_cv_scores[(c, n)])
    for c in RF_CRITERIA for n in RF_N_ESTIMATORS
}
best_rf_key = max(rf_means, key=rf_means.get)
best_rf_crit = best_rf_key.split('(')[0]
best_rf_n = int(best_rf_key.rstrip(')').split('(')[1])
best_rf_model = RandomForestClassifier(
    criterion=best_rf_crit, n_estimators=best_rf_n,
    random_state=RANDOM_STATE, n_jobs=-1,
)

# Stacking
stacking_means = {n: np.mean(stacking_cv_scores[n]) for n in STACKING_N_VALUES}
best_stacking_n = max(stacking_means, key=stacking_means.get)
best_stacking_model = StackingClassifier(
    estimators=generate_committee(best_stacking_n, make_simple_mlp),
    final_estimator=LogisticRegression(random_state=RANDOM_STATE),
    cv=3, n_jobs=-1,
)

# Voting
voting_means = {n: np.mean(voting_cv_scores[n]) for n in VOTING_N_VALUES}
best_voting_n = max(voting_means, key=voting_means.get)
best_voting_model = VotingClassifier(
    estimators=generate_committee(best_voting_n, make_simple_mlp),
    voting='soft', n_jobs=-1,
)

print(f"Melhor Bagging: {best_bagging_key} (F1={bagging_means[best_bagging_key]:.4f})")
print(f"Melhor RF:      {best_rf_key} (F1={rf_means[best_rf_key]:.4f})")
print(f"Melhor Stacking: Stacking({best_stacking_n}) (F1={stacking_means[best_stacking_n]:.4f})")
print(f"Melhor Voting:  Voting({best_voting_n}) (F1={voting_means[best_voting_n]:.4f})")


Melhor Bagging: NB(15) (F1=0.6649)
Melhor RF:      gini(100) (F1=0.6706)
Melhor Stacking: Stacking(5) (F1=0.6805)
Melhor Voting:  Voting(20) (F1=0.6815)


In [49]:
# Evaluate the 4 best models on all bases to get CV scores for ANOVA
final_models = {
    'Bagging': best_bagging_model,
    'Random Forest': best_rf_model,
    'Stacking': best_stacking_model,
    'Voting': best_voting_model,
}

FINAL_CACHE = RESULTS_DIR + 'final_comparison.pkl'
if os.path.exists(FINAL_CACHE):
    with open(FINAL_CACHE, "rb") as f:
        final_cv_scores = pickle.load(f)
    print("Resultados finais carregados do cache.")
else:
    final_cv_scores = {}
    for name, model in final_models.items():
        all_scores = []
        for ds in DATASETS:
            xs, ys = all_data[ds]
            scores = cross_val_score(model, xs, ys, cv=10, scoring=SCORING, n_jobs=-1)
            all_scores.append(scores)
        final_cv_scores[name] = np.concatenate(all_scores)
        print(f"  {name} OK")

    with open(FINAL_CACHE, "wb") as f:
        pickle.dump(final_cv_scores, f)
    print("Cache salvo.")

# Display summary
final_summary = pd.DataFrame({
    name: [np.mean(scores), np.std(scores)]
    for name, scores in final_cv_scores.items()
}, index=['F1-macro Médio', 'Desv. Pad.']).T
final_summary


  Bagging OK
  Random Forest OK
  Stacking OK
  Voting OK
Cache salvo.


,F1-macro Médio,Desv. Pad.
Bagging,0.664882,0.095233
Random Forest,0.670552,0.082155
Stacking,0.680619,0.091552
Voting,0.681484,0.093271


### Teste estatístico final

Verificando diferença estatística entre os 4 melhores modelos.

In [50]:
final_names = list(final_cv_scores.keys())
final_groups = [final_cv_scores[n] for n in final_names]

if any(np.std(g) < 1e-10 for g in final_groups):
    print("AVISO: Variância zero detectada.")
else:
    f_stat, p_val = f_oneway(*final_groups)
    print(f"ANOVA: F={f_stat:.4f}, p={p_val:.6f}")

    if p_val < 0.05:
        print("Diferença significativa entre os modelos!")
        res = tukey_hsd(*final_groups)
        means = {n: np.mean(final_cv_scores[n]) for n in final_names}
        best_name = max(means, key=means.get)
        best_idx = final_names.index(best_name)
        
        # Show pairwise p-values
        print("\nP-valores pairwise (Tukey HSD):")
        for i, n1 in enumerate(final_names):
            for j, n2 in enumerate(final_names):
                if i < j:
                    p = abs(res.pvalue[i, j])
                    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
                    print(f"  {n1:15s} vs {n2:15s}: p={p:.4f} {sig}")
        
        # Check if best is significantly better than all others
        all_sig = all(
            abs(res.pvalue[best_idx, i]) < 0.05
            for i in range(len(final_names)) if i != best_idx
        )
        if all_sig:
            print(f"\n>>> Melhor modelo global: {best_name} (F1={means[best_name]:.4f}) <<<")
        else:
            # Pick by complexity among non-significantly different candidates
            complexity_order = ['NB', 'k-NN', 'DT', 'MLP', 'Random Forest',
                               'Bagging', 'Voting', 'Stacking']
            candidates = [n for i, n in enumerate(final_names)
                          if abs(res.pvalue[best_idx, i]) >= 0.05 or i == best_idx]
            
            def comp_key(name):
                for i, c in enumerate(complexity_order):
                    if c in name:
                        return i
                return 99
            best_name = min(candidates, key=comp_key)
            print(f"\n>>> Melhor modelo global (por complexidade): {best_name} <<<")
    else:
        print("Sem diferença significativa entre os modelos.")
        # Complexity-based choice
        means = {n: np.mean(final_cv_scores[n]) for n in final_names}
        best_mean_val = max(means.values())
        candidates = [n for n, v in means.items() if abs(v - best_mean_val) < 0.01]
        complexity_order = ['NB', 'k-NN', 'DT', 'MLP', 'Random Forest',
                           'Bagging', 'Voting', 'Stacking']
        def comp_key(name):
            for i, c in enumerate(complexity_order):
                if c in name:
                    return i
            return 99
        best_name = min(candidates, key=comp_key)
        print(f">>> Melhor modelo global (por complexidade): {best_name} <<<")


ANOVA: F=0.9368, p=0.422666
Sem diferença significativa entre os modelos.
>>> Melhor modelo global (por complexidade): Voting <<<


## Conclusão

Este notebook implementou 4 métodos de comitê de classificadores (Bagging, Random Forest, Stacking e Voting) sobre 12 bases de características extraídas de imagens de raças de cães e gatos. Para cada método, foram testadas diferentes configurações de hiperparâmetros e realizadas análises estatísticas (ANOVA + Tukey HSD) para selecionar o melhor modelo. Por fim, os melhores modelos de cada método foram comparados para determinar a melhor abordagem de comitê para este problema.